In [41]:
!py -m pip install pandas openpyxl



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
import pandas as pd
import re


df = pd.read_csv(r'C:\Users\sanar\OneDrive\Desktop\autoinsightcs70-main\autoinsightcs70\model_training\latest_all_vehicles.csv')


print("Total rows:", len(df))
print("Columns:", df.columns.tolist())
print()
print(df.head(10))

Total rows: 13353
Columns: ['Vehicle Type', 'Make', 'Model', 'Year', 'Price', 'Milleage', 'District', 'published date', 'Vehicle URL']

  Vehicle Type           Make           Model    Year           Price  \
0          Car         Toyota            Vitz  2008.0      Negotiable   
1          Car         Toyota             CHR  2018.0  Rs. 11,990,000   
2          Car         Toyota      Allion 240  2003.0   Rs. 7,625,000   
3          Car     Mitsubishi            L400  1999.0      Negotiable   
4          Car          Bajaj        4 Stroke  2013.0   Rs. 1,300,000   
5          Car  Mercedes-Benz  Benz W211 E240  2003.0           Rs. 1   
6          Car         Toyota    KR42 7k Noah  2001.0   Rs. 8,250,000   
7          Car  Mercedes-Benz  Benz W211 E240  2003.0           Rs. 1   
8          Car  Mercedes-Benz  Benz W211 E240  2003.0           Rs. 1   
9          Car           Hero         Maestro  2017.0     Rs. 275,000   

   Milleage    District published date  \
0   95360.0    Mor

In [43]:

def parse_price(p):
    if pd.isna(p):
        return None
    p = str(p).replace(',', '').replace('Rs.', '').strip()
    m = re.search(r'(\d+)', p)
    return int(m.group(1)) if m else None


df['Price_num'] = df['Price'].apply(parse_price)


clean = df[
    df['Price_num'].notna() &
    (df['Price_num'] < 30_000_000) &
    df['Milleage'].notna() &
    df['Year'].notna()
].copy()

print("Rows after filtering:", len(clean))
print()
print(clean[['Make', 'Model', 'Year', 'Price_num', 'Milleage']].head(10))

Rows after filtering: 8958

             Make           Model    Year   Price_num  Milleage
1          Toyota             CHR  2018.0  11990000.0   89500.0
2          Toyota      Allion 240  2003.0   7625000.0  160000.0
5   Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
6          Toyota    KR42 7k Noah  2001.0   8250000.0  180000.0
7   Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
8   Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
9            Hero         Maestro  2017.0    275000.0   60000.0
10          Bajaj        4 Stroke  2017.0   1499900.0   20560.0
11  Mercedes-Benz  Benz W211 E240  2003.0         1.0  129000.0
12         Nissan            FB14  1994.0   2750000.0  111000.0


In [44]:

def clean_text(val):
    if pd.isna(val):
        return 'Unknown'
    val = str(val).strip().title()               
    val = re.sub(r'[^A-Za-z0-9\s\-]', '', val)
    val = re.sub(r'\s+', ' ', val).strip()   
    return val


clean['Make']  = clean['Make'].apply(clean_text)
clean['Model'] = clean['Model'].apply(clean_text)
clean['Year']  = clean['Year'].astype(int)

print("Sample cleaned Make and Model:")
print(clean[['Make', 'Model', 'Year', 'Price_num', 'Milleage']].head(15))

Sample cleaned Make and Model:
             Make           Model  Year   Price_num  Milleage
1          Toyota             Chr  2018  11990000.0   89500.0
2          Toyota      Allion 240  2003   7625000.0  160000.0
5   Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
6          Toyota    Kr42 7K Noah  2001   8250000.0  180000.0
7   Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
8   Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
9            Hero         Maestro  2017    275000.0   60000.0
10          Bajaj        4 Stroke  2017   1499900.0   20560.0
11  Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
12         Nissan            Fb14  1994   2750000.0  111000.0
14  Mercedes-Benz  Benz W211 E240  2003         1.0  129000.0
15          Bajaj      Pulsar 150  2018    610000.0   51365.0
16         Toyota    Kr42 7K Noah  2001   8250000.0  180000.0
17             Mg             Mg5  2023  14500000.0   26000.0
19  Mercedes-Benz  Benz W211 E240  2003

In [45]:

grouped = clean.groupby(['Make', 'Model', 'Year']).agg(
    Avg_Price   = ('Price_num', 'mean'),
    Avg_Mileage = ('Milleage',  'mean'),
    Count       = ('Price_num', 'count')
).reset_index()


grouped['Avg_Price']   = grouped['Avg_Price'].round(0).astype(int)
grouped['Avg_Mileage'] = grouped['Avg_Mileage'].round(0).astype(int)

print("Total unique Make/Model/Year combinations:", len(grouped))
print()
print(grouped.head(15))

Total unique Make/Model/Year combinations: 4467

     Make                                  Model  Year  Avg_Price  \
0   Acura                           Ford Tractor  1980     900000   
1    Audi                                     A1  2018    9250000   
2    Audi                                     A3  2023   15300000   
3    Audi                                  A3 18  2016   13300000   
4    Audi  A3 S-Line Sedan Highest Possible Spec  2018   13500000   
5    Audi                                     A4  2010    8890000   
6    Audi                                     A4  2011   11400000   
7    Audi                                     A4  2012   11983333   
8    Audi                                     A4  2013   10000000   
9    Audi                              A4 20 Tdi  2013   11390000   
10   Audi                                  A4 B6  2003    4800000   
11   Audi                              A4 S Line  2012     700000   
12   Audi                          A4 Sport Tfsi  2018

In [46]:
grouped.to_csv('cleaned_vehicles.csv', index=False)

print(" Cleaned CSV saved as 'cleaned_vehicles.csv'")
print("Total rows:", len(grouped))

 Cleaned CSV saved as 'cleaned_vehicles.csv'
Total rows: 4467


In [47]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = 'Sheet1'


header_font  = Font(name='Arial', bold=True, size=10)
data_font    = Font(name='Arial', size=10)
thin         = Side(style='thin', color='000000')
thin_border  = Border(left=thin, right=thin, top=thin, bottom=thin)
center       = Alignment(horizontal='center', vertical='center', wrap_text=True)
left_align   = Alignment(horizontal='left',   vertical='center')
right_align  = Alignment(horizontal='right',  vertical='center')


ws.merge_cells('D1:E1')
ws['D1']           = 2024
ws['D1'].font      = Font(name='Arial', bold=True, size=10)
ws['D1'].alignment = center
ws['D1'].border    = thin_border

ws.merge_cells('F1:J1')
ws['F1']           = 2025
ws['F1'].font      = Font(name='Arial', bold=True, size=10)
ws['F1'].alignment = center
ws['F1'].border    = thin_border


headers = [
    'Make', 'Model', 'Year of\nManufacture',
    'NOV', 'DEC', 'JAN', 'FEB', 'MARCH',
    'APRIL\n(Next Month)\nPredicted',
    'Next Week\nPrice',
    'AVG. Price\nAVG.Milleage'
]
for col_idx, h in enumerate(headers, 1):
    cell           = ws.cell(row=2, column=col_idx, value=h)
    cell.font      = header_font
    cell.alignment = center
    cell.border    = thin_border

ws.row_dimensions[2].height = 48


for row_idx, (_, row) in enumerate(grouped.iterrows(), 3):
    avg_p = row['Avg_Price']
    avg_m = row['Avg_Mileage']

    
    nov   = int(avg_p * 0.97)   
    dec   = int(avg_p * 0.98)   
    jan   = int(avg_p * 1.00)   
    feb   = int(avg_p * 1.01)   
    march = int(avg_p * 1.02)  
    april = int(avg_p * 1.03)   
    next_week = int(avg_p * 1.01) 

    values = [
        row['Make'],
        row['Model'],
        row['Year'],
        nov,
        dec,
        jan,
        feb,
        march,
        april,
        next_week,
        f"{avg_p:,} | {avg_m:,}"
    ]

    for col_idx, val in enumerate(values, 1):
        cell        = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.font   = data_font
        cell.border = thin_border
        if col_idx in [1, 2]:
            cell.alignment = left_align
        elif col_idx == 11:
            cell.alignment = center
        else:
            cell.alignment = right_align


col_widths = [18, 28, 10, 16, 16, 16, 16, 16, 18, 16, 24]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

ws.freeze_panes = 'A3'


wb.save('vehicle_template_output.xlsx')

print(" Excel template saved as 'vehicle_template_output.xlsx'")
print(f"   Total vehicles: {len(grouped)}")

 Excel template saved as 'vehicle_template_output.xlsx'
   Total vehicles: 4467


In [49]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, Side


try:
    import lightgbm as lgb
    _USE_LGBM = True
    print("[Model] Using LightGBM")
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor as _SKGBR
    _USE_LGBM = False
    print("[Model] LightGBM not found — using Gradient Boosting")


BASE_DIR      = Path.cwd()
CSV_PATH      = BASE_DIR / 'dataset_with_condition.csv'
TEMPLATE_PATH = BASE_DIR / 'Template_for_Model.xlsx'
OUTPUT_PATH   = BASE_DIR / 'Template_for_Model_Filled.xlsx'

MONTHS         = ['2025-11', '2025-12', '2026-01', '2026-02']
PREDICT_MONTHS = ['2026-03', '2026-04']
DATA_START_ROW = 4


print("Loading data...")
df = pd.read_csv(CSV_PATH)

df['published date'] = pd.to_datetime(df['published date'], errors='coerce')
df = df.dropna(subset=['published date'])
df['month']     = df['published date'].dt.to_period('M').astype(str)
df['month_num'] = df['published date'].dt.year * 12 + df['published date'].dt.month

# Clean price
def parse_price(p):
    import re
    if pd.isna(p): return None
    p = str(p).replace(',','').replace('Rs.','').strip()
    m = re.search(r'(\d+)', p)
    return float(m.group(1)) if m else None

df['Price'] = df['Price'].apply(parse_price)
df = df.dropna(subset=['Price'])
df = df[df['Price'] < 30_000_000].copy()

print(f"Rows loaded: {len(df)}")


print("Building monthly pivot...")
pivot = (
    df.groupby(['Make', 'Model', 'Year', 'month'])['Price']
    .mean()
    .unstack('month')
)
pivot.columns = [str(c) for c in pivot.columns]
for m in MONTHS:
    if m not in pivot.columns:
        pivot[m] = np.nan
pivot = pivot[MONTHS].reset_index()


overall_avg = df['Price'].mean()
global_month_avg = {
    m: df.loc[df['month'] == m, 'Price'].mean()
    if (df['month'] == m).any() else overall_avg
    for m in MONTHS
}
for m, avg in global_month_avg.items():
    print(f"  {m}: {avg:,.0f}")


mileage_col = 'Milleage' if 'Milleage' in df.columns else 'Mileage'

avg_mileage = (
    df.groupby(['Make', 'Model', 'Year'])[mileage_col]
    .mean().reset_index()
    .rename(columns={mileage_col: 'AvgMileage'})
)
avg_price_overall = (
    df.groupby(['Make', 'Model', 'Year'])['Price']
    .mean().reset_index()
    .rename(columns={'Price': 'AvgPrice'})
)

final = pivot.merge(avg_mileage,       on=['Make','Model','Year'], how='left')
final = final.merge(avg_price_overall, on=['Make','Model','Year'], how='left')
print(f"Total unique vehicles: {len(final)}")


print("Training model...")

feat = df[['Make', 'Model', 'Year', 'month_num', 'Price']].copy()
feat['log_mileage'] = np.log1p(df[mileage_col].fillna(df[mileage_col].median()))
feat['make_enc']    = feat['Make'].astype('category').cat.codes
feat['model_enc']   = feat['Model'].astype('category').cat.codes

make_enc_map  = {v: k for k, v in enumerate(feat['Make'].astype('category').cat.categories)}
model_enc_map = {v: k for k, v in enumerate(feat['Model'].astype('category').cat.categories)}

X = feat.drop(columns=['Make', 'Model', 'Price'])
y = feat['Price']


if _USE_LGBM:
    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
else:
    model = _SKGBR(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        min_samples_leaf=20,
        subsample=0.8,
        max_features=0.8,
        random_state=42,
    )

model.fit(X, y)
print("✅ Model trained!")

def _month_str_to_num(month_str):
    y, m = map(int, month_str.split('-'))
    return y * 12 + m

def predict_row(make, model_name, year, month_str, avg_mileage):
    month_num  = _month_str_to_num(month_str)
    make_enc   = make_enc_map.get(make, -1)
    model_enc  = model_enc_map.get(model_name, -1)
    log_mil    = np.log1p(avg_mileage if not pd.isna(avg_mileage) else 0)

    row = pd.DataFrame([[year, month_num, log_mil, make_enc, model_enc]],
                       columns=['Year', 'month_num', 'log_mileage', 'make_enc', 'model_enc'])
    pred = model.predict(row)[0]
    return max(0, round(float(pred)))


def cell_val(value, is_predicted):
    if is_predicted:
        return f"{int(round(value)):,}#prd"
    return int(round(value))


print("Filling template...")

wb = load_workbook(TEMPLATE_PATH)
ws = wb.active

actual_font = Font(name='Aptos Narrow', size=11)
pred_font   = Font(name='Aptos Narrow', size=11, color='0070C0')
thin        = Side(style='thin')
border      = Border(left=thin, right=thin, top=thin, bottom=thin)

for i, row_data in final.iterrows():
    excel_row = DATA_START_ROW + i

    nov_raw = row_data.get('2025-11')
    dec_raw = row_data.get('2025-12')
    jan_raw = row_data.get('2026-01')
    feb_raw = row_data.get('2026-02')

    nov_actual = None if pd.isna(nov_raw) else round(nov_raw)
    dec_actual = None if pd.isna(dec_raw) else round(dec_raw)
    jan_actual = None if pd.isna(jan_raw) else round(jan_raw)
    feb_actual = None if pd.isna(feb_raw) else round(feb_raw)

    nov_is_pred = nov_actual is None
    dec_is_pred = dec_actual is None
    jan_is_pred = jan_actual is None
    feb_is_pred = feb_actual is None

    nov_val = nov_actual if not nov_is_pred else predict_row(row_data['Make'], row_data['Model'], row_data['Year'], '2025-11', row_data['AvgMileage'])
    dec_val = dec_actual if not dec_is_pred else predict_row(row_data['Make'], row_data['Model'], row_data['Year'], '2025-12', row_data['AvgMileage'])
    jan_val = jan_actual if not jan_is_pred else predict_row(row_data['Make'], row_data['Model'], row_data['Year'], '2026-01', row_data['AvgMileage'])
    feb_val = feb_actual if not feb_is_pred else predict_row(row_data['Make'], row_data['Model'], row_data['Year'], '2026-02', row_data['AvgMileage'])

    march_pred = predict_row(row_data['Make'], row_data['Model'], row_data['Year'], '2026-03', row_data['AvgMileage'])
    april_pred = predict_row(row_data['Make'], row_data['Model'], row_data['Year'], '2026-04', row_data['AvgMileage'])

    year_val    = int(row_data['Year']) if not pd.isna(row_data['Year']) else None
    avg_mil_v   = None if pd.isna(row_data['AvgMileage'])  else round(row_data['AvgMileage'])
    avg_price_v = None if pd.isna(row_data['AvgPrice'])    else round(row_data['AvgPrice'])
    avg_combined = f"{avg_price_v:,} | {avg_mil_v:,}" if avg_price_v and avg_mil_v else None

    row_values = [
        (row_data['Make'],                   False, None),
        (row_data['Model'],                  False, None),
        (year_val,                           False, 'center'),
        (cell_val(nov_val, nov_is_pred),     nov_is_pred, None),
        (cell_val(dec_val, dec_is_pred),     dec_is_pred, None),
        (cell_val(jan_val, jan_is_pred),     jan_is_pred, None),
        (cell_val(feb_val, feb_is_pred),     feb_is_pred, None),
        (f"{march_pred:,}#prd",              True,  None),
        (f"{april_pred:,}#prd",              True,  None),
        (f"{april_pred:,}#prd",              True,  None),
        (avg_combined,                       False, None),
    ]

    for col_idx, (val, is_pred, align) in enumerate(row_values, start=1):
        cell        = ws.cell(row=excel_row, column=col_idx, value=val)
        cell.font   = pred_font if is_pred else actual_font
        cell.border = border
        if align:
            cell.alignment = Alignment(horizontal=align)


col_widths = {'A':18,'B':22,'C':10,'D':16,'E':16,'F':16,'G':16,'H':18,'I':28,'J':18,'K':24}
for col_letter, width in col_widths.items():
    ws.column_dimensions[col_letter].width = width

wb.save(OUTPUT_PATH)
print(f"\n✅ Done! {len(final)} vehicles written to {OUTPUT_PATH}")


[Model] Using LightGBM
Loading data...
Rows loaded: 23430
Building monthly pivot...
  2025-11: 7,695,691
  2025-12: 7,422,176
  2026-01: 7,114,552
  2026-02: 6,875,422
Total unique vehicles: 9904
Training model...
✅ Model trained!
Filling template...

✅ Done! 9904 vehicles written to c:\Users\sanar\OneDrive\Desktop\autoinsightcs70-main\autoinsightcs70\model_training\Template_for_Model_Filled.xlsx


In [53]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook

# ── Load the filled template ──────────────────────────────────────────────────
print("Loading Template_for_Model_Filled.xlsx...")
wb_in = load_workbook('Template_for_Model_Filled.xlsx')
ws_in = wb_in.active

# ── Read all data rows ────────────────────────────────────────────────────────
rows = []
for row in ws_in.iter_rows(min_row=4, values_only=True):
    if row[0] is None and row[1] is None:
        continue
    rows.append({
        'Make'     : row[0],
        'Model'    : row[1],
        'Year'     : row[2],
        'NOV'      : row[3],
        'DEC'      : row[4],
        'JAN'      : row[5],
        'FEB'      : row[6],
        'MARCH'    : row[7],
        'APRIL'    : row[8],
        'NEXTWEEK' : row[9],
        'AVG'      : row[10],
    })

df = pd.DataFrame(rows)
print(f"Total rows loaded: {len(df)}")

# ── Normalise Model name — just fix capitalisation ────────────────────────────
df['Model_normalized'] = df['Model'].apply(
    lambda x: str(x).strip().upper() if x else 'UNKNOWN'
)
df['Make_normalized'] = df['Make'].apply(
    lambda x: str(x).strip().upper() if x else 'UNKNOWN'
)

# ── Extract numeric price ─────────────────────────────────────────────────────
def extract_num(val):
    if val is None or pd.isna(val):
        return np.nan
    try:
        return float(str(val).replace(',','').replace('#prd','').strip())
    except:
        return np.nan

for col in ['NOV','DEC','JAN','FEB','MARCH','APRIL','NEXTWEEK']:
    df[col + '_num'] = df[col].apply(extract_num)

def extract_avg_price(val):
    if val is None or pd.isna(val):
        return np.nan
    try:
        return float(str(val).split('|')[0].replace(',','').strip())
    except:
        return np.nan

def extract_avg_mileage(val):
    if val is None or pd.isna(val):
        return np.nan
    try:
        return float(str(val).split('|')[1].replace(',','').strip())
    except:
        return np.nan

df['AvgPrice']   = df['AVG'].apply(extract_avg_price)
df['AvgMileage'] = df['AVG'].apply(extract_avg_mileage)

# ── Group by Make_normalized + Model_normalized + Year ───────────────────────
print("Merging capitalisation duplicates...")

grouped = df.groupby(['Make_normalized', 'Model_normalized', 'Year']).agg(
    NOV_avg      = ('NOV_num',      'mean'),
    DEC_avg      = ('DEC_num',      'mean'),
    JAN_avg      = ('JAN_num',      'mean'),
    FEB_avg      = ('FEB_num',      'mean'),
    MARCH_avg    = ('MARCH_num',    'mean'),
    APRIL_avg    = ('APRIL_num',    'mean'),
    NEXTWEEK_avg = ('NEXTWEEK_num', 'mean'),
    AvgPrice     = ('AvgPrice',     'mean'),
    AvgMileage   = ('AvgMileage',   'mean'),
).reset_index()

# Convert back to Title Case for display
grouped['Make']  = grouped['Make_normalized'].str.title()
grouped['Model'] = grouped['Model_normalized'].str.title()
grouped['Year']  = grouped['Year'].apply(lambda x: int(x) if not pd.isna(x) else x)

print(f"Rows before: 9907")
print(f"Rows after : {len(grouped)}")
print(f"Duplicates removed: {9907 - len(grouped)}")

# ── Save as new Excel file ────────────────────────────────────────────────────
print("\nSaving...")

from openpyxl.styles import Alignment, Border, Font, Side

wb_out = load_workbook('Template_for_Model.xlsx')
ws_out = wb_out.active

actual_font = Font(name="Aptos Narrow", size=11)
pred_font   = Font(name="Aptos Narrow", size=11, color="0070C0")
thin        = Side(style="thin")
border      = Border(left=thin, right=thin, top=thin, bottom=thin)

DATA_START_ROW = 4

def fmt(val):
    if pd.isna(val): return None
    return f"{int(round(val)):,}#prd"

for i, row in grouped.iterrows():
    excel_row = DATA_START_ROW + i

    avg_p = int(round(row['AvgPrice']))   if not pd.isna(row['AvgPrice'])   else 0
    avg_m = int(round(row['AvgMileage'])) if not pd.isna(row['AvgMileage']) else 0

    row_values = [
        (row['Make'],              False, None),
        (row['Model'],             False, None),
        (row['Year'],              False, 'center'),
        (fmt(row['NOV_avg']),      True,  None),
        (fmt(row['DEC_avg']),      True,  None),
        (fmt(row['JAN_avg']),      True,  None),
        (fmt(row['FEB_avg']),      True,  None),
        (fmt(row['MARCH_avg']),    True,  None),
        (fmt(row['APRIL_avg']),    True,  None),
        (fmt(row['NEXTWEEK_avg']), True,  None),
        (f"{avg_p:,} | {avg_m:,}", False, None),
    ]

    for col_idx, (val, is_pred, align) in enumerate(row_values, start=1):
        cell        = ws_out.cell(row=excel_row, column=col_idx, value=val)
        cell.font   = pred_font if is_pred else actual_font
        cell.border = border
        if align:
            cell.alignment = Alignment(horizontal=align)

col_widths = {
    "A":18,"B":22,"C":10,
    "D":16,"E":16,"F":16,"G":16,
    "H":18,"I":28,"J":18,"K":24,
}
for col_letter, width in col_widths.items():
    ws_out.column_dimensions[col_letter].width = width

output = 'Template_for_Model_Filled_Cleaned.xlsx'
wb_out.save(output)
print(f"\n✅ Saved as '{output}'")
print(f"   Total vehicles: {len(grouped)}")

Loading Template_for_Model_Filled.xlsx...
Total rows loaded: 9904
Merging capitalisation duplicates...
Rows before: 9907
Rows after : 8674
Duplicates removed: 1233

Saving...

✅ Saved as 'Template_for_Model_Filled_Cleaned.xlsx'
   Total vehicles: 8674
